In [1]:
import numpy as np
from numpy.linalg import eigh
import torch

**A note on the code below:**

When we conduct the diagonalization of $V\Lambda V^*$, we wish to use `eigh` instead of `eig` since `eigh` uses a faster algorithm that **assumes the matrix is Hermitian** $(A = A^*)$. That is to say, we should only pass in Hermitian matrices into `eigh`. This poses a problem since our normal matrix $V\Lambda V^*$ is not Hermitian. Below is what $V\Lambda V^*$ is when $P=3$:

\begin{bmatrix}
-0.5 & 0.866 & 1.118 \\
-0.866 & -0.5 & 1.936 \\
-1.118 & -1.936 & -0.5
\end{bmatrix}

However, $K = V\Lambda V^* - \frac{1}{2}I$ is *skew-Hermitian* $(-A = A^*)$. This means $V\Lambda V^*$ is of the form:
$$V\Lambda V^* = \alpha I + K$$
Where $\alpha\in\mathbb{R}$ and $K$ is skew-Hermitian. Now, observe that:
$$V\Lambda V^*v = \lambda v \iff Kv = (\lambda - \alpha)v$$
That is to say that $V\Lambda V^*$ and its skew part $K$ **share the same eigenvectors**, albeit their eigenvalues are shifted by $\alpha$.

If we multiply both sides by $-i$:
$$-iV\Lambda V^* = -i\alpha I + -iK$$
We then have:
$$(-iV\Lambda V^*)v = \mu v \iff (-iK)v = (\mu + i\alpha)v$$
Which is to say the **eigenvectors of $V\Lambda V^*$, $K$, $-iV\Lambda V^*$, and $-iK$ are all the same!** Their eigenvalues are simply scaled $(\mu = -i\lambda)$. Lucky for us, $-iK$ is Hermitian, so we will seek to (indirectly) use it as the input for `eigh`.

To see why $-iK$ is Hermitian, first note that it is a fact that multiplying a skew-Hermitian matrix by $-i$ yields a Hermitian matrix:
$$\text{Sketch proof: } -K=K^* \implies -iK = i(-K) = iK^* = (-iK)^*$$

<!-- So we have the following:
$$-iV\Lambda V^* = \underbrace{-i\alpha I}_{\text{skew-Hermitian}} + \underbrace{-iK}_{\text{Hermitian}} $$ -->

What `eigh` really does is "take the lower triangular part of the matrix and assumes that the upper triangular part is defined by the symmetry of the matrix" [(stackoverflow)](https://stackoverflow.com/questions/45434989/numpy-difference-between-linalg-eig-and-linalg-eigh). The [documentation](https://numpy.org/doc/2.3/reference/generated/numpy.linalg.eigh.html) for `eigh` actually states in regards to the optional second argument `UPLO`:
> Specifies whether the calculation is done with the lower triangular part of a (‘L’, default) or the upper triangular part (‘U’). Irrespective of this value only the real parts of the diagonal will be considered in the computation to preserve the notion of a Hermitian matrix. It therefore follows that the imaginary part of the diagonal will always be treated as zero.

Here we note that $-iV\Lambda V^*$ has a *purely imaginary diagonal*, so if we pass it into `eigh`, it will effectively be treated as $-iK$ since $K$ is just $V\Lambda V^*$ with the diagonal removed. So we leverage this technical nuance to avoid computing $K$ and just pass in `VΛV * -1j` to yield the desired eigenvector matrix $V$.




In [2]:
def DPLR_HiPPO(P):
    
    # HiPPO
    P_arr = np.arange(P)
    M = np.sqrt(2*P_arr + 1) # creates a P-dimensional array according to HiPPO-LegS
    A = -np.tril(np.outer(M,M)) + np.diag(P_arr) # A_nk in Appendix C.1 of S4 paper
    
    # DPLR
    Q = np.sqrt(P_arr + 0.5) # low-rank array (i.e. vector) part of the "low rank correction". See "adding..." in C.1 of S4 paper
    VΛV = A + np.outer(Q,Q)
    Λ_real = np.diagonal(VΛV) # this is just a P-dimensional array of -0.5's
    Λ_imag, V = eigh(VΛV * -1j)
    Q_tilde = V.conj() @ Q
    
    B = np.sqrt(2 * P_arr + 1.0)
    B_tilde = V.conj() @ B   
    
    return (
        # note the imaginary eigenvalues are being multiplied by i to account for the -i scaling above
        # recall that \mu = -i\lambda, so to recover \lambda we just multiply by i
        torch.tensor(np.asarray(Λ_real + 1j * Λ_imag), dtype=torch.complex64),
        torch.tensor(np.asarray(V), dtype=torch.complex64),
        torch.tensor(np.asarray(Q_tilde)),
        torch.tensor(np.asarray(B_tilde))
    ) 
    

If you wish to have confirmation that this diagonalization works, run the cell below for any value of `P` you desire.

In [ ]:
P = 3
P_arr = np.arange(P)
M = np.sqrt(2*P_arr + 1) # creates a P-dimensional array according to HiPPO-LegS
A = -np.tril(np.outer(M,M)) + np.diag(P_arr) # A_nk in Appendix C.1 of S4 paper

# NPLR
Q = np.sqrt(P_arr + 0.5) # part of the "low rank correction".
VΛV = A + np.outer(Q,Q)
Λ_real = np.diagonal(VΛV) # this is just a P-dimensional array of -0.5's
Λ_imag, V = eigh(VΛV * -1j)

Λ = np.diag(Λ_real + 1j*Λ_imag)
print(np.allclose(A,V @ Λ @ V.conj().T - np.outer(Q,Q), atol=1e-4, rtol=1e-4))


Now we set up the forward ZOH variant of the LLH SSM layer.

In [3]:
from typing import Optional, Tuple
import torch.nn as nn
import torch.nn.functional as F

A lot of technical notes for the `Forward_LLH` class below:

#### `_init_A()`:
**Ensuring $\mathfrak{R}(\Lambda)<0$:**

Eventually we will need to discretize $A$ as $\bar{A}=e^{\Lambda\Delta t}$. This raises a risk of numerical instability if the real parts of $\Lambda$ are not negative. Of course, gradient descent is unable to "ignore" negative values during the backpropagation process, so instead a transformation is performed to ensure that only negative eigenvalues are considered.

Instead of making `Λ_P.real` a parameter itself, we represent it via:
$$\theta = \log(-\mathfrak{R}(\Lambda))$$
Note here that $\Lambda$ is initalized as a diagonal matrix whose real parts are all $-0.5$, so logarithm above is well-defined. Gradient descent is performed on $\theta\in\mathbb{R}$, and once $\Lambda$ must be re-constructed, we simply apply:
$$\mathfrak{R}(\Lambda) = -e^{\theta}$$
Hence, we are guaranteed negative real parts for $\Lambda$

**Input-dependent dynamics and `Δ_net`:**

`relative_time` has to do with whether we want to rescale $\Lambda$ at each event time based on the current input $u_{t_i}$ (à la [Mamba](https://openreview.net/forum?id=tEYskw1VY2)). This is the "input-dependent dynamics" section of the paper (see B.3 of S2P2 paper.) $\Delta_i$ is a time-scaling diagonal matrix that changes the relative timescale of each channel of the state's influence (a scaling for each of the $P$ channels). Large values suggest time should pass quickly for that channel, suggesting faster convergence to steady-state, and smaller values will cause the model to retain the influence that prior events have on future ones due to the slower perceived passage of time. If `relative_time = True`, we make this a learnable network via $$\Delta_i = \text{diag(Softplus}(W * u_{t_i} + b))$$ and call it `Δ_net`. It is then multiplied by $\Lambda$ to get a rescaled diagonal matrix $\Lambda_i$

**Initializing the bias of $\Delta_i$ in a numerically sounds way:**

Note that $$\text{Softplus}(x) = \log(1 + e^x)$$ Recall that we are multiplying $\Delta_i=\text{diag(Softplus}(W * u_{t_i} + b))$ by $\Lambda$, so at initialization we want the softplus's output to be close to 1, so $\Lambda_i$ is close to $\Lambda$. Now, we already have that the input $u$ at initialization is 0 (in the first layer), so the $W*u$ term in the
softplus will be 0. so we are left with initializing the bias such that $\text{Softplus}(b) = 1$
We can show what this looks like algebraically:
$$\text{Softplus}(b) = \log(1 + e^b) = 1 \rightarrow b = \log(e^1 - 1)$$
The bias vector is initialized as a $P$-dimensional tensor of ones, then add `log(-expm1(-1))`$=\log(-(e^{-1}-1))$ to each element,
which is algebraically equivalent to the $\log(e^1 - 1)$ desired.

Why is this done? Because the expression $e^x - 1$ is numerically unstable (e.g. catastrophic cancellation) for small values of $x$. $x + log(1-e^{-x})$ is more numerically stable for small values of x than $\log(e^x - 1)$.

#### `_init_B()`:

Although there is a theoretical HiPPO B, it was found in [Gu et al., 2022](https://papers.neurips.cc/paper_files/paper/2022/file/e9a32fade47b906de908431991440f7c-Paper-Conference.pdf) on the beginning of page 6 that it was not necessary to initialize $B$ according to the HiPPO, only $A$, due to its dominant effect on the dynamics of the SSM. However, allowing $B$ to be learned freely still allows for some material gain, so they opt to initialize $B$ randomly with Xavier normal initialization (while still multiplying it by the eigenvector matrix $V$.)

#### `forward()`:

**Broadcasting the initial state `x_0_P` across all the batches:**

The initial state tensor `x_0_P` is only $P$-dimensional, but we need it to be broacast across all the batches as a $B\times P$-dimensional tensor. This is accomplished by appending dimensions via `view()` and setting those dimensions via `expand()`.

The `view()` method returns a new tensor with the same data but of a different shape, without copying memory (see [documentation](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html) and [stackoverflow](https://stackoverflow.com/questions/42479902/what-does-view-do-in-pytorch).)

The `*` operator is known as the "packing/unpacking" operator. The `view()` method takes arguments one number at a time, not in a list, so since we want to feed the numbers from the list into `view()` we unpack it with `*` (i.e. `foo(*[1,2,3])` is equivalent to `foo(1,2,3)`)

`expand()` does the final broadcasting without copying data (see [documentation](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.expand.html)). So if the current tensor has shape `(1,1,64)`, if we call `.expand(32,100,-1)`, it produces a tensor of size `(32,100,64)` the `-1` denotes "keep this dimension the original size", whereas the "1" dimensions are broadcasted to the indicated sizes.

**Checking the dimensions between input signal and mark impulse:**

The `assert all()` block found is just a shape check between the input signal and mark impulse. the `zip()` method pairs elements of one list with the elements of the other list in order. so if `x.shape = y.shape = (B,N,H)`, then `zip(x.shape,y.shape)` will produce pairs `(B,B)` `(N,N)` and `(H,H)` the `for` loop checks to see that each of these dimensions match, and the `assert all()` checks that every output of the `for` loop is True. (`all()` returns True if every element is True, False otherwise) `assert` will throw an error if it returns false.

**`pre_norm` vs. `post_norm`:**

Most models in the current literature discuss and analyze the implications of applying $\text{LayerNorm}$ at the end of the current pass, or the beginning of the next pass. This has material implications in the sense that in the post-normalization approach:
$$u^{(l+1)} = \text{LayerNorm}(\sigma(y^{(l)}) + u^{(l)})$$
the normalization is applied *after* adding the residual stream $u^{(l)}$. This means the distribution of the added residual will also be shifted. When compared to the pre-normalization approach:
$$u^{(l+1)} = \text{LayerNorm}(\sigma(y^{(l)})) + u^{(l)}$$
The residual stream's distribution (across the $H$ features) is not considered.

In discussion with the one of the model's authors, they stated
>Empirically, we found post-norm works well in our model, likely because the depth of our architecture remains within a range where the benefits of better signal preservation outweigh the optimization instabilities typically seen in much deeper models.



In [ ]:
class Forward_LLH(nn.Module):
    
    def __init__(
        self,
        P: int,
        H: int,
        dropout_rate: float = 0.0,
        pre_norm: bool = True,
        post_norm: bool = False,
        is_first_layer: bool = False,
        relative_time: bool = False
    ):
        super(Forward_LLH, self).__init__()
        
        # inscribe config file parameters as class attributes
        self.P = P
        self.H = H
        self.dropout_rate = dropout_rate
        self.pre_norm = pre_norm
        self.post_norm = post_norm
        self.is_first_layer = is_first_layer
        self.relative_time = relative_time
        
        self.act_func = nn.Sequential(nn.GELU(), nn.Dropout(p=self.dropout_rate)) # D(GELU())
        self.norm = nn.LayerNorm(self.H) # we will normalize over the H features of the input and output
        
        # the multiplication by .001 is in line with the practice of avoiding uncontrolled intial state magnitude that can 
        # destabilize training. 
        self.x_0_P = nn.Parameter(torch.complex(torch.randn(self.P), torch.randn(self.P)) * 1e-3, 
                                  requires_grad=True)
        
        self._init_ssm_params()
        
    def _init_ssm_params(self):
        self._init_A()
        if not self.is_first_layer:
            self._init_B()
        self._init_C()
        if not self.is_first_layer:
            self._init_D()
        self._init.E()
        
    def _init_A(self):        
        Λ_P, V_PP, _, _ = DPLR_HiPPO(self.P)
        
        self.V_PP = V_PP
        self.V_star_PP = V_PP.conj().T
        
        # the real part of Λ is initialized to -0.5 across all P channels so it is guaranteed that log is receiving 
        # a positive input here
        self.log_neg_real_Λ_P = nn.Parameter((-Λ_P.real).log()) 
        self.imag_Λ_P = nn.Parameter(Λ_P.imag)
        
        if self.relative_time:
            self.Δ_net = nn.Linear(self.H, self.P, bias=True)
            with torch.no_grad():
                self.Δ_net.weight.copy_(nn.init.xavier_normal_(self.Δ_net.weight))
                
                bias = torch.ones(self.P)
                bias += torch.log(-torch.expm1(-bias))
                self.Δ_net.bias.copy_(bias)
        else:
            self.log_step_size_P = nn.Parameter(torch.zeros(size=(self.P)), requires_grad=False) # this may be unneeded
            
    @property
    def Λ_P(self):
        return torch.complex(-torch.exp(self.log_neg_real_Λ_P), self.imag_Λ_P)
    
    def _init_B(self):
        B = nn.init.xavier_normal_(torch.zeros((self.P, self.H)))
        B_tilde_PH = self.V_star_PP @ B.type(torch.complex64) # TODO: paper says we multiply by negative V*?
        self.B_tilde_PH = nn.Parameter(B_tilde_PH)
        
    def _init_C(self):       
        C = nn.init.xavier_normal_(torch.zeros((self.H, self.P)))
        C_tilde_HP = C.type(torch.complex64) @ self.V_PP
        self.C_tilde_HP = nn.Parameter(C_tilde_HP)
        
    def _init_D(self):        
        D_H = torch.zeros(self.H)
        nn.init.normal_(D_H, std=1.0)
        self.D_H = nn.Parameter(D_H)
        
    def _init_E(self):        
        E = nn.init.xavier_normal_(torch.zeros((self.P, self.H))) # R = H
        E_tilde_PH = self.V_star_PP @ E.type(torch.complex64)
        self.E_tilde_PH = nn.Parameter(E_tilde_PH)
        
    def compute_impulse(self,α_H):
        # although the paper defines α as a RxK (we set R = H) matrix, we are only taking a column vector from this matrix
        tilde_Eα_P = torch.einsum("ph,...h->...p", self.E_tilde_PH, α_H.type(torch.complex64))
        return tilde_Eα_P
    
    def get_Λ_i(self, right_u_NH, shift_u=True):
        if self.relative_time and (right_u_NH is not None): # right_u is null for first layer
            if shift_u:
                right_u_NH = F.pad(right_u_NH[..., :-1,:], (0,0,1,0)) # TODO: maybe explain this
            Λ_i_NP = F.softplus(self.Δ_net(right_u_NH)) * self.Λ_P
            return {"Λ_i_NP": Λ_i_NP} # these keys will be used in _ssm in order to determine the dimension
        else:
            if self.relative_time: # relative_time is true but it's the first layer (right_u is null)
                Λ_i_P = F.softplus(self.Δ_net.bias) * self.Λ_P # there is no u to multiply the weight matrix by
            else:
                Λ_i_P = self.Λ_P
            return {"Λ_i_P": Λ_i_P} # these keys will be used in _ssm in order to determine the dimension
        
    def forward(
        self,
        left_u_NH: Optional[torch.Tensor],
        right_u_NH: Optional[torch.Tensor],
        α_NH: torch.Tensor,
        dt_N: torch.Tensor, # [0, t1-t0, ..., t_N-t_{N-1}]
        x_0_P: Optional[torch.Tensor] = None # this argument may be omitted
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        
        *leading_dims, _, _ = α_NH.shape # leading_dims will typically just be batch size B
        num_leading_dims = len(leading_dims)
        
        # the state is P-dimensional, but it needs to be broacast across the B sequences 
        # basically we just need to reformat initial_state_P as a [...,P] tensor (most likely a [B,P] tensor)
        if x_0_P is None:
            x_0_P = self.x_0_P.view(*[1 for _ in range(num_leading_dims)], -1).expand(*leading_dims, -1)
        normed_left_u_NH = left_u_NH
        normed_right_u_NH = right_u_NH
        if normed_left_u_NH is not None:
            # checking that all the dimensions match
            assert all(u_d == a_d for u_d, a_d in zip(normed_left_u_NH.shape, α_NH.shape))
            if self.pre_norm:
                normed_left_u_NH = self.norm(normed_left_u_NH) # normed_left_u_NH is not actually norm'd until now
        if normed_right_u_NH is not None:
            # checking that all the dimensions match
            assert all(u_d == a_d for u_d, a_d in zip(normed_right_u_NH.shape, α_NH.shape))
            if self.pre_norm:
                normed_right_u_NH = self.norm(normed_right_u_NH) # normed_right_u_NH is not actually norm'd until now
        
        # this _ssm() call is what actually carries out the bulk of of Algorithm 1
        right_x_NP, left_y_NH, right_y_NH = self._ssm(
            left_u_NH = normed_left_u_NH,
            right_u_NH = normed_right_u_NH,
            tilde_Eα_NP = self.compute_impulse(α_NH),
            dt_N = dt_N,
            x_0_P = x_0_P
        )
        
        next_layer_left_u_NH = next_layer_right_u_NH = None
        if left_y_NH is not None:
            next_layer_left_u_NH = self.act_func(left_y_NH) + (left_u_NH if left_u_NH is not None else 0.0)
            if self.post_norm:
                next_layer_left_u_NH = self.norm(next_layer_left_u_NH)
        if right_y_NH is not None:
            next_layer_right_u_NH = self.act_func(right_y_NH) + (right_u_NH if right_u_NH is not None else 0.0)
            if self.post_norm:
                next_layer_right_u_NH = self.norm(next_layer_right_u_NH)
        
        return right_x_NP, next_layer_left_u_NH, next_layer_right_u_NH
    
    def _ssm(
        self,
        left_u_NH: Optional[torch.Tensor],
        right_u_NH: Optional[torch.Tensor],
        tilde_Eα_NP: torch.Tensor,
        dt_N: torch.Tensor,
        x_0_P: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        
        *leading_dims, N, P = tilde_Eα_NP.shape
        
        Λ_i = self.get_Λ_i(right_u_NH, shift_u=True)
        if "Λ_i_P" in Λ_i: # we use this key check instead of shape check because both cases end in the dimension of P
            # TODO: check why there is no ellipsis in second argument of einsum. no batch?
            Λ_dt_NP = torch.einsum("...n,p->...np", dt_N, Λ_i["Λ_i_P"]) # element-wise product but a diagonal matrix
        else:
            # scaling each row of Λ_i_NP by associated element of dt_N
            Λ_dt_NP = torch.einsum("...n,...np->...np", dt_N, Λ_i["Λ_i_NP"]) 
        
        # compute left-hand Du    
        if left_u_NH is not None:
            left_Du_NH = torch.einsum("...nh,h->...nh", left_u_NH, self.D_H)
        else:
            assert self.is_first_layer
            left_Du_NH = 0.0
            
        # computing right-hand Bu and Du
        if right_u_NH is not None:
            right_u_NH = F.pad(right_u_NH[..., :-1, :], (0, 0, 1, 0))
            right_Bu_NP = torch.einsum( # Equation 19 of S2P2 paper
                "...np,ph,...nh->...np",
                Λ_dt_NP.exp() - 1, # Λ_bar - 1
                self.B_tilde_PH,
                right_u_NH.type(torch.complex64)
            )
            right_Du_NH = torch.einsum("...nh,h->...nh", right_u_NH, self.D_H)
        else:
            assert self.is_first_layer
            right_Du_NH = right_Bu_NP =  0.0
            
        # parallel scan as per (Heinsen 2023) and https://github.com/PeaBrane/mamba-tiny/blob/master/scans.py
        log_impulse_Np1_P = torch.concat((x_0_P.unsqueeze(-2), right_Bu_NP + tilde_Eα_NP), dim = -2).log()
        Λ_dt_star = F.pad(Λ_dt_NP.cumsum(-2), (0, 0, 1, 0))
        right_log_x_NP = torch.logcumsumexp(log_impulse_Np1_P - Λ_dt_star, -2) + Λ_dt_star
        right_x_NP = right_log_x_NP.exp() # Contains initial_state_P in index 0
        left_x_NP = Λ_dt_NP.exp() * right_x_NP[..., :-1, :] + right_Bu_NP # this is Equation (15)
        right_x_NP = right_x_NP[..., 1:, :] # finally we exponentiate, ignoring the first dimension padded earlier
        
        left_y_NH = 2 * torch.einsum("hp,...np->...nh", self.C_tilde_HP, left_x_NP).real + left_Du_NH
        right_y_NH = 2 * torch.einsum("hp,...np->...nh", self.C_tilde_HP, right_x_NP).real + right_Du_NH
        
        return right_x_NP, left_y_NH, right_y_NH
        
    def get_x_left_limit(
        self,
        right_x_P: torch.Tensor,
        dt_G: torch.Tensor,
        current_right_u_H: torch.Tensor
    ) -> torch.Tensor:
        
        if current_right_u_H is not None and self.pre_norm:
            current_right_u_H = self.norm(current_right_u_H)
            
        Λ_i = self.get_Λ_i(current_right_u_H, shift_u = False) # input signal u should already been shifted
        if "Λ_i_P" in Λ_i:
            Λ_bar_GP = torch.exp(torch.einsum("...g,p->...gp", dt_G, Λ_i["Λ_i_P"]))
        else:
            Λ_bar_GP = torch.exp(torch.einsum("...g,...p->...gp", dt_G, Λ_i["Λ_i_NP"]))
            
        Λ_bar_x_GP = torch.einsum("...p,...gp->...gp", right_x_P, Λ_bar_GP)
        
        if current_right_u_H is None:
            assert self.is_first_layer
            return Λ_bar_x_GP
        else: # add Bu to impulse
            if self.pre_norm:
                current_right_u_H = self.norm(current_right_u_H)
            impulse_GP = torch.einsum(
                "...gp,ph,...h->...gp", 
                Λ_bar_GP - 1.0, 
                self.B_tilde_PH, 
                current_right_u_H.type(torch.complex64))
            return Λ_bar_x_GP + impulse_GP
    
    def depth_pass(
        self,
        current_left_x_P: torch.Tensor,
        current_left_u_H: Optional[torch.Tensor]
    ) -> torch.Tensor:
        
        if current_left_u_H is not None:
            if self.pre_norm:
                normed_u_H = self.norm(current_left_u_H)
            else:
                normed_u_H = current_left_u_H
            left_Du_H = torch.einsum("...h,h->...h", normed_u_H, self.D_H)
        else:
            assert self.is_first_layer
            left_Du_H = 0.0
            
        y_H = 2 * torch.einsum("...p,hp->...h", current_left_x_P, self.C_tilde_HP).real + left_Du_H
        
        if self.post_norm:
            new_u_H = self.norm(self.act_func(y_H) + (current_left_u_H if current_left_u_H is not None else 0.0))
        else:
            new_u_H = self.act_func(y_H) + (current_left_u_H if current_left_u_H is not None else 0.0)
            
        return new_u_H
        

In [30]:
Lambda = torch.randint(0,5,(3,2))
dt = torch.randint(0,5,(3,))
print(Lambda)
print(dt)

tensor([[4, 0],
        [1, 4],
        [0, 1]])
tensor([2, 1, 2])


In [31]:
torch.einsum("n,np->...np", dt, Lambda)

tensor([[8, 0],
        [1, 4],
        [0, 2]])